In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import zipfile
import os
# import geopandas as gpd

## Prepare economic data

In [ ]:
df_2018 = pd.read_csv('./data_2018/18incd.csv')
df_2022 = pd.read_csv('./data_2023/22incdoh.csv')

import pandas as pd

def clean_economic_data(df, separator=" "):
    
    # 1. Drop first 3 rows
    df = df.iloc[3:].copy()
    
    # 2. Drop completely empty rows
    df = df.dropna(how="all")
    
    # 3. Combine first two columns into one
    state_col, district_col = df.columns[:2]
    df["CD116"] = (
            df[state_col].astype(str).str.strip()
            + " "
            + df[district_col].astype(int).astype(str).str.zfill(2)
        )
    cols = ["CD116"] + [c for c in df.columns if c != "CD116"]
    df = df[cols]
    df = df.drop(columns=[state_col, district_col, "Unnamed: 3"])

    # Rename columns
    new_column_names=["CD116", "Household Income", "Percentage"]
    df.columns = new_column_names
    df = df.dropna(subset=["Household Income"])

    # Change to wide format
    df_wide = df.pivot(index="CD116", columns="Household Income", values="Percentage").copy()
    df_wide = df_wide.reset_index()
    
    return df_wide

# Apply function
df_2018_clean = clean_economic_data(df_2018)
df_2022_clean = clean_economic_data(df_2022)
df_2022_clean = df_2022_clean.rename(columns={'CD116': 'CD119'})

# Export
df_2018_clean.to_csv('./data_2018/demographic_data/18incd_clean.csv')
df_2022_clean.to_csv('./data_2023/22incdoh_clean.csv')

# 2018 data

Data to be used from 2018 to train the model

In [ ]:
# Function for cleaning data
def clean_cd_data(file_path, rename_dict, columns):
    #Read file
    data = pd.read_csv(file_path,
                          header=2,
                          skiprows=[3,4])
    #Rename columns
    data = data.rename(columns=rename_dict)
    #Filter out other states data
    states_list = ['Ohio', 'Indiana', 'Michigan', 'Pennsylvania', 'Wisconsin', 'Missouri']
    data = data[data['State name'].isin(states_list)]
    #Specify columns
    data = data[columns]
    #Add column for join
    data['CD116FP'] = data['Congressional district'].astype(int)
    return data

# Get new voter turnout data
turnout_file = "./data_2018/demographic_data/voter_turnout_2018.csv"
turnout_dict = {'Voting rate3': 'Voting rate'}
turnout_columns = ['State name', 'Congressional district', 'Voting rate']
voter_turnout = clean_cd_data(turnout_file, turnout_dict,turnout_columns)

# Age data
age_cd_file = "./data_2018/demographic_data/table02a_age_2018.csv"
age_cd_dict = {'Unnamed: 8': '18-29',
    'Unnamed: 12': '30-44',
    'Unnamed: 16': '45-64',
    'Unnamed: 20': '65 and older'}
age_cd_columns = ['State name', 'Congressional district', '18-29', '30-44', '45-64', '65 and older']
age_cd_gdf = clean_cd_data(age_cd_file, age_cd_dict, age_cd_columns)

# Sex & Poverty data
sex_poverty_cd_file = "./data_2018/demographic_data/table02b_sex_poverty_2018.csv"
sex_poverty_cd_dict = {'Unnamed: 8': 'Men',
    'Unnamed: 12': 'Women',
    'Unnamed: 18': 'In Poverty'}
sex_poverty_cd_columns = ['State name', 'Congressional district', 'Men', 'Women', 'In Poverty']
sex_poverty_cd_gdf = clean_cd_data(sex_poverty_cd_file, sex_poverty_cd_dict, sex_poverty_cd_columns)

# Education data
education_cd_file = "./data_2018/demographic_data/table02c_education_2018.csv"
education_cd_dict = {'Unnamed: 8': 'Less than 9th grade',
    'Unnamed: 12': '9th to 12 Grade, no diploma',
    'Unnamed: 16': 'High school graduate',
    'Unnamed: 36': 'High school or more',
    'Unnamed: 40': 'Bachelors or more'}
education_cd_columns = ['State name', 'Congressional district', 'Less than 9th grade', '9th to 12 Grade, no diploma', 'High school graduate', 'High school or more', 'Bachelors or more']
education_cd_gdf = clean_cd_data(education_cd_file, education_cd_dict, education_cd_columns)
# Making new column combining those that did not finish high school
education_cd_gdf['Did not finish high school'] = (
    education_cd_gdf['Less than 9th grade'] + education_cd_gdf['9th to 12 Grade, no diploma']
)

education_cd_gdf['High school or less'] = (
    education_cd_gdf['Less than 9th grade'] + education_cd_gdf['9th to 12 Grade, no diploma'] + education_cd_gdf['High school graduate']
)

education_cd_gdf = education_cd_gdf.drop(
    ['Less than 9th grade', '9th to 12 Grade, no diploma'],
    axis=1
)

# Race data
race_cd_file = "./data_2018/demographic_data/table02d_race_2018.csv"
race_cd_dict = {'Unnamed: 8': 'White',
    'Unnamed: 12': 'Black',
    'Unnamed: 16': 'Asian',
    'Unnamed: 36': 'Hispanic'}
race_cd_columns = ['State name', 'Congressional district', 'White', 'Black', 'Asian', 'Hispanic']
race_cd_gdf = clean_cd_data(race_cd_file, race_cd_dict, race_cd_columns)
# Make sure data is numerical
race_cd_gdf.replace('N', np.nan, inplace=True)
cols_to_convert = ['White', 'Black', 'Asian', 'Hispanic']
for col in cols_to_convert:
    race_cd_gdf[col] = pd.to_numeric(race_cd_gdf[col], errors='coerce')

# Economic data

# Merge data to get a dataset called 'df' with all varaibles:
merge_cols = ['State name', 'Congressional district', 'CD116FP']
df = voter_turnout.merge(age_cd_gdf, on=merge_cols, how='inner') \
            .merge(sex_poverty_cd_gdf, on=merge_cols, how='inner') \
            .merge(education_cd_gdf, on=merge_cols, how='inner') \
            .merge(race_cd_gdf, on=merge_cols, how='inner')
df['18-44'] = df['18-29'] + df['30-44']
df['CD116'] = df['State name'] + ' ' + df['CD116FP'].astype(str).str.zfill(2)


# Merge urbanization df
urbanization_df = pd.read_csv('./data_2018/urbanization_2018.csv')
df = df.merge(
    urbanization_df[['CD116', 'urbanization_pct']],
    on='CD116',
    how='left'
)

# Merge econ
econ_df =  pd.read_csv('./data_2018/demographic_data/18incd_clean.csv')
econ_cd_columns = ["Under $1", "$1 under $10,000",	"$10,000 under $25,000", "$25,000 under $50,000", "$50,000 under $75,000", "$75,000 under $100,000", "$100,000 under $200,000",	"$200,000 under $500,000", "$500,000 or more" ]
df = df.merge(econ_df, on='CD116', how='inner').copy()

# Choose Select columns
df = df[['CD116', 'Voting rate', '18-44', '45-64',
       '65 and older', 'In Poverty', 
       # 'Women',
       'Did not finish high school', 'High school or less', 'Bachelors or more',
       'White', 'Black', 'Asian', 'Hispanic',
       'Under $1', '$1 under $10,000', '$10,000 under $25,000',
       '$25,000 under $50,000', '$50,000 under $75,000', '$500,000 or more',
       '$75,000 under $100,000',
       '$100,000 under $200,000', '$200,000 under $500,000',
       'urbanization_pct']]

# df.loc[:, df.columns != 'Voter Data'] = df.loc[:, df.columns != 'Voter Data'].round(1)

# # Further refine columns
# df['Non-white'] = 100 - df['White']
# df['45 and older'] = df['65 and older'] + df['45-64']

# df = df[['CD116', 'Voting rate', 'Bachelors or more', '45 and older', 'In Poverty', 'Non-white', ]]

# Export as csv
df.to_csv('./data_2018/cd_2018.csv', index=False)

# 2022 - 2024 data

Most recent data for model to be used on

In [ ]:
# Read CSV
df = pd.read_csv('./data_2023/Ohio_District_all_2023.csv')

# Step 1: Melt the dataframe to long format
df_long = pd.melt(
    df, 
    id_vars=['Title'], 
    var_name='CD119FP', 
    value_name='Value'
)

# Step 2: Pivot so that Title values become columns
df_wide = df_long.pivot(
    index='CD119FP', 
    columns='Title', 
    values='Value'
).reset_index()

df_wide['CD119FP'] = df_wide['CD119FP'].str.extract(r'(\d+)').astype(int)
for col in df_wide.columns.difference(['CD119FP']):
    df_wide[col] = pd.to_numeric(df_wide[col].astype(str).str.replace(',', ''), errors='coerce')

# Sex, poverty, & education variables
df_wide['Women'] = (df_wide['Female']) / (df_wide['Female'] + df_wide['Male']) * 100
df_wide['Did not finish high school'] = 100 - df_wide['Percent high school graduate or higher']
df_wide.rename(columns={'Poverty Rate': 'In Poverty', 'Percent bachelor\'s degree or higher': 'Bachelors or more'},
                inplace=True)
df_wide.drop(['Female', 'Male', 'Percent high school graduate or higher',
              'Less than 9th grade', 'Total Population 25 years and over', '9th to 12th grade, no diploma'], axis=1, inplace=True)

# Age variables
# Isolate Age variables to calculate age
age_df = df_wide[['CD119FP', 'Total population', 'Under 5 years','5 to 9 years', '10 to 14 years', '15 to 19 years', 
       '20 to 24 years', '25 to 34 years', '35 to 44 years', '45 to 54 years',
       '55 to 59 years', '60 to 64 years', '18 years and over', '65 years and over']].copy()
age_df['18-19'] = age_df['Under 5 years'] + age_df['5 to 9 years'] + age_df['10 to 14 years'] + age_df['15 to 19 years'] - age_df['Total population'] + age_df['18 years and over']
age_df['Voting Population'] = age_df['18 years and over']
age_df['18-44'] = (age_df['18-19'] + age_df['20 to 24 years'] + age_df['25 to 34 years'] + age_df['35 to 44 years']) / age_df['Voting Population'] * 100
age_df['45-64'] = (age_df['45 to 54 years'] + age_df['55 to 59 years'] + age_df['60 to 64 years']) / age_df['Voting Population'] * 100
age_df['65 and older'] = age_df['65 years and over'] / age_df['Voting Population'] * 100
age_df = age_df[['CD119FP', '18-44', '45-64', '65 and older']]

# Race variables
race_df = df_wide[['CD119FP', 'Total population', 'White', 'Black or African American',
                   'Asian', 'Hispanic or Latino (of any race)']].copy()
race_df.rename(columns={'Black or African American': 'Black', 'Hispanic or Latino (of any race)': 'Hispanic'},
                inplace=True)
race_df['White'] = race_df['White'] / race_df['Total population'] * 100
race_df['Black'] = race_df['Black'] / race_df['Total population'] * 100
race_df['Asian'] = race_df['Asian'] / race_df['Total population'] * 100
race_df['Hispanic'] = race_df['Hispanic'] / race_df['Total population'] * 100

# Merge final dataset
cd = df_wide.copy()
# Drop race & age columns
cd.drop(['Under 5 years','5 to 9 years', '10 to 14 years', '15 to 19 years', 
       '20 to 24 years', '25 to 34 years', '35 to 44 years', '45 to 54 years',
       '55 to 59 years', '60 to 64 years', '18 years and over', '65 years and over',
       'White', 'Black or African American','Asian', 'Hispanic or Latino (of any race)'],axis=1, inplace=True)
# Merge back age df
cd = cd.merge(
    age_df[['CD119FP', '18-44', '45-64', '65 and older']],
    on='CD119FP',
    how='left'
)
# Merge back race df
cd = cd.merge(
    race_df[['CD119FP', 'White', 'Black', 'Asian', 'Hispanic']],
    on='CD119FP',
    how='left'
)
cd['CD119'] = 'Ohio ' + cd['CD119FP'].astype(str).str.zfill(2)

# Merge urbanization df
urbanization_df = pd.read_csv('./data_2023/urbanization_2024.csv')
cd = cd.merge(
    urbanization_df[['CD119FP', 'urbanization_pct']],
    on='CD119FP',
    how='left'
)

# Merge econ
econ_df =  pd.read_csv('./data_2023/22incdoh_clean.csv')
cd = cd.merge(econ_df, on='CD119', how='inner').copy()

# Final columns included are same as 2018 dataset
cd = cd[['CD119', '18-44', '45-64',
       '65 and older', 'In Poverty', 'Did not finish high school',
       # 'Women',
       'Bachelors or more', 'White', 'Black', 'Asian', 'Hispanic',
       'Under $1', '$1 under $10,000', '$10,000 under $25,000',
       '$25,000 under $50,000', '$50,000 under $75,000', '$500,000 or more',
       '$75,000 under $100,000',
       '$100,000 under $200,000', '$200,000 under $500,000',
       'urbanization_pct']]
cd = cd.round(1)

# Further choose columns
# cd['Non-white'] = 100 - cd['White']
# cd['45 and older'] = cd['65 and older'] + cd['45-64']
# cd = cd[['CD119', 'Bachelors or more', '45 and older', 'In Poverty', 'Non-white', ]]

cd.to_csv('./data_2023/cd_2023.csv', index=False)
